# EDA ראשוני — דאטאסט קמעונאי ישראלי
**מטרה:** להכיר את הנתונים לפני שנבנה את הסוכנים

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# הגדרות תצוגה
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

print('ספריות נטענו בהצלחה!')

## 1. טעינת הדאטא

In [ ]:
df = pd.read_csv('../data/raw/israeli_retail.csv', encoding='utf-8-sig')
print(f'גודל הדאטאסט: {df.shape[0]:,} שורות x {df.shape[1]} עמודות')
df.head()

In [ ]:
# סוגי עמודות + ערכים חסרים
print('סוגי עמודות:')
print(df.dtypes)
print(f'\nערכים חסרים: {df.isnull().sum().sum()}')

## 2. סטטיסטיקות בסיסיות

In [ ]:
print('=== סיכום כספי ===')
print(f"סה\"כ מכירות:  ₪{df['מכירות_ש'].sum():>12,.0f}")
print(f"סה\"כ רווח:    ₪{df['רווח_ש'].sum():>12,.0f}")
print(f"אחוז רווח:    {df['רווח_ש'].sum()/df['מכירות_ש'].sum()*100:.1f}%")
print(f"\nממוצע עסקה:  ₪{df['מכירות_ש'].mean():>9,.1f}")
print(f"חציון עסקה:  ₪{df['מכירות_ש'].median():>9,.1f}")

## 3. מכירות לפי קטגוריה

In [ ]:
cat_sales = df.groupby('קטגוריה')['מכירות_ש'].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# גרף 1 — מכירות לפי קטגוריה
cat_sales.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('מכירות לפי קטגוריה (₪)', fontsize=13)
axes[0].set_xlabel('סה"כ מכירות (₪)')

# גרף 2 — אחוז מכירות (פאי)
cat_sales.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('חלוקת מכירות לפי קטגוריה', fontsize=13)
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../data/processed/chart_categories.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. מכירות לפי רשת

In [ ]:
chain_stats = df.groupby('רשת').agg(
    מכירות=('מכירות_ש', 'sum'),
    רווח=('רווח_ש', 'sum'),
    עסקאות=('מספר_הזמנה', 'count')
).round(0)
chain_stats['אחוז_רווח'] = (chain_stats['רווח'] / chain_stats['מכירות'] * 100).round(1)
chain_stats.sort_values('מכירות', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
chain_stats['מכירות'].sort_values().plot(kind='barh', ax=ax, color='coral')
ax.set_title('מכירות לפי רשת קמעונאית (₪)', fontsize=13)
ax.set_xlabel('סה"כ מכירות (₪)')
plt.tight_layout()
plt.savefig('../data/processed/chart_chains.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. מכירות לפי חודש (טרנד)

In [ ]:
df['תאריך'] = pd.to_datetime(df['תאריך'])
monthly = df.groupby(df['תאריך'].dt.to_period('M'))['מכירות_ש'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
monthly.plot(ax=ax, color='green', linewidth=2, marker='o', markersize=4)
ax.set_title('טרנד מכירות חודשי 2022–2024', fontsize=13)
ax.set_xlabel('חודש')
ax.set_ylabel('מכירות (₪)')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/processed/chart_monthly_trend.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. השפעת ימי השבוע

In [ ]:
day_order = ['ראשון', 'שני', 'שלישי', 'רביעי', 'חמישי', 'שישי', 'שבת']
day_sales = df.groupby('יום_בשבוע')['מכירות_ש'].sum().reindex(day_order)

colors = ['#e74c3c' if d == 'שבת' else 'steelblue' for d in day_order]
fig, ax = plt.subplots(figsize=(10, 5))
day_sales.plot(kind='bar', ax=ax, color=colors)
ax.set_title('מכירות לפי יום בשבוע (אדום = שבת)', fontsize=13)
ax.set_xlabel('יום')
ax.set_ylabel('סה"כ מכירות (₪)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../data/processed/chart_weekday.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. השפעת חגים

In [ ]:
holiday_sales = df[df['חג'] != ''].groupby('חג')['מכירות_ש'].sum().sort_values(ascending=False)
regular_avg   = df[df['חג'] == '']['מכירות_ש'].mean()

print(f'ממוצע מכירות ביום רגיל: ₪{regular_avg:,.0f}')
print('\nמכירות לפי חג:')
print(holiday_sales.to_string())

## 8. מפת חום — קטגוריה לפי עיר

In [ ]:
pivot = df.pivot_table(values='מכירות_ש', index='עיר', columns='קטגוריה', aggfunc='sum', fill_value=0)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('מפת חום — מכירות לפי עיר וקטגוריה (₪)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/chart_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. תובנות ראשוניות

✍️ **כתוב כאן את התובנות שלך אחרי שראית את הגרפים:**

1. הקטגוריה המובילה היא: ___
2. הרשת עם הכי הרבה מכירות: ___
3. החודש הכי חזק בשנה: ___
4. ההשפעה הכי גדולה על מכירות: ___
5. תובנה מפתיעה שגיליתי: ___